# SON Algorithm

## Learning Objectives

1. **State** the SON (Savasere-Omiecinski-Navathe) theorem and its proof via pigeonhole
2. **Explain** why using local threshold $s/p$ guarantees no false negatives
3. **Describe** the two-pass structure and its MapReduce implementation
4. **Compare** SON to A-Priori in terms of passes, memory, and correctness
5. **Implement** SON with configurable partition count


## Problem Statement

### A-Priori's Memory Constraint

A-Priori requires all candidate pairs to fit in RAM during Pass 2. For very large datasets (billions of baskets), even candidate singletons may not fit in RAM across all partitions.

### SON's Approach

Partition the dataset into $p$ chunks, each small enough to fit in RAM. Run A-Priori on each chunk independently using a *lower* support threshold $s/p$ (to avoid missing any globally frequent itemset). Take the union of all locally frequent itemsets as the candidate set $C$ for a second global pass.

**Key property:** Only 2 passes over disk. Easily parallelised in MapReduce.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <defs><marker id="arr" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#999"/></marker></defs>
  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">SON Algorithm: Partition → Local → Global</text>

  <!-- Step 1 -->
  <rect x="20" y="38" width="230" height="130" rx="5" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="135" y="58" text-anchor="middle" fill="#4e79a7" font-size="11" font-weight="bold">Pass 1 — Per Partition</text>
  <text x="135" y="78" text-anchor="middle" fill="#333" font-size="11">Split data into p chunks</text>
  <text x="135" y="96" text-anchor="middle" fill="#333" font-size="11">Each chunk fits in RAM</text>
  <text x="135" y="116" text-anchor="middle" fill="#333" font-size="11">Run A-Priori on chunk i</text>
  <text x="135" y="134" text-anchor="middle" fill="#333" font-size="11">with threshold s/p</text>
  <text x="135" y="152" text-anchor="middle" fill="#555" font-size="10">(lower threshold = local support)</text>

  <line x1="250" y1="103" x2="298" y2="103" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Step 2 -->
  <rect x="300" y="38" width="230" height="130" rx="5" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="415" y="58" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">Candidate Set C</text>
  <text x="415" y="78" text-anchor="middle" fill="#333" font-size="11">Union of locally frequent</text>
  <text x="415" y="96" text-anchor="middle" fill="#333" font-size="11">itemsets across all chunks</text>
  <text x="415" y="116" text-anchor="middle" fill="#555" font-size="11">Key theorem:</text>
  <text x="415" y="134" text-anchor="middle" fill="#333" font-size="11">globally frequent ⟹</text>
  <text x="415" y="152" text-anchor="middle" fill="#333" font-size="11">locally frequent in ≥1 chunk</text>

  <line x1="530" y1="103" x2="578" y2="103" stroke="#999" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- Step 3 -->
  <rect x="580" y="38" width="220" height="130" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="690" y="58" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">Pass 2 — Global Count</text>
  <text x="690" y="78" text-anchor="middle" fill="#333" font-size="11">Scan full dataset once</text>
  <text x="690" y="96" text-anchor="middle" fill="#333" font-size="11">Count only candidates in C</text>
  <text x="690" y="116" text-anchor="middle" fill="#333" font-size="11">Keep itemsets with</text>
  <text x="690" y="134" text-anchor="middle" fill="#333" font-size="11">global count ≥ s</text>
  <text x="690" y="152" text-anchor="middle" fill="#555" font-size="10">No false negatives by theorem</text>

  <!-- Correctness theorem -->
  <rect x="20" y="186" width="780" height="60" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="410" y="206" text-anchor="middle" fill="#59a14f" font-size="12" font-weight="bold">SON Theorem (No False Negatives)</text>
  <text x="410" y="226" text-anchor="middle" fill="#333" font-size="11">If itemset I is globally frequent (count ≥ s), then it must be locally frequent in at least one chunk.</text>
  <text x="410" y="242" text-anchor="middle" fill="#555" font-size="10">Proof: If count(I) ≥ s and there are p chunks of equal size, then count(I) in some chunk ≥ s/p (by pigeonhole).</text>

  <!-- Example -->
  <rect x="20" y="260" width="780" height="90" rx="5" fill="#f5f5f5" stroke="#ccc" stroke-width="1"/>
  <text x="410" y="280" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Example: 6 baskets, s=3, p=2 chunks of 3</text>
  <text x="200" y="300" text-anchor="middle" fill="#4e79a7" font-size="11">Chunk 1 (local s=1.5 → 2):</text>
  <text x="200" y="316" text-anchor="middle" fill="#333" font-size="11">{A,B,C}, {A,B}, {A,C}</text>
  <text x="200" y="332" text-anchor="middle" fill="#333" font-size="11">Local frequent: {A}=3✓ {B}=2✓ {C}=2✓ {A,B}=2✓</text>
  <text x="590" y="300" text-anchor="middle" fill="#f28e2b" font-size="11">Chunk 2 (local s=2):</text>
  <text x="590" y="316" text-anchor="middle" fill="#333" font-size="11">{A,C,D}, {B,D}, {A,D}</text>
  <text x="590" y="332" text-anchor="middle" fill="#333" font-size="11">Local frequent: {A}=2✓ {D}=3✓ {A,D}=2✓</text>
  <text x="410" y="346" text-anchor="middle" fill="#555" font-size="10">C = {{A},{B},{C},{D},{A,B},{A,C},{A,D}} → Pass 2 counts globally → truly frequent = {{A},{A,C}} etc.</text>
</svg>
'''
display(SVG(svg))


## SON Theorem

**Theorem:** If itemset $I$ is globally frequent (count $\geq s$ over all $n$ baskets), then $I$ is locally frequent (count $\geq s/p$) in at least one chunk.

**Proof:** Suppose $I$ is globally frequent but locally infrequent in every chunk. Then its count in every chunk $< s/p$, so its global count $< p \cdot (s/p) = s$ — contradiction. $\square$

**Consequence:** The union $C$ of all locally frequent itemsets contains every globally frequent itemset. Pass 2 only needs to count candidates in $C$ — no false negatives.

**False positives:** $C$ may contain itemsets that are locally but not globally frequent. Pass 2 eliminates them by counting globally.

### MapReduce Implementation

**Map Pass 1:** Each mapper receives one partition and runs A-Priori with threshold $s/p$. Emits `(itemset, 1)` for each locally frequent itemset.

**Reduce Pass 1:** Collect all emitted itemsets (union = $C$).

**Map Pass 2:** Each mapper counts candidates $C$ in its partition. Emits `(itemset, local_count)`.

**Reduce Pass 2:** Sum counts for each itemset; keep those $\geq s$.


## Algorithm Steps

1. **Partition** data into $p$ equal-sized chunks fitting in RAM
2. **Pass 1:** For each chunk, run A-Priori with local threshold $\lceil s/p \rceil$; collect candidate set $C = \bigcup_i L_i$
3. **Pass 2:** Scan all baskets once; count each candidate $c \in C$; keep those with global count $\geq s$
4. **Output** globally frequent itemsets


In [ ]:
from collections import defaultdict
from itertools import combinations


def apriori_one_chunk(baskets, min_support):
    """Run A-Priori on a single in-memory chunk, return frequent itemsets."""
    freq = {}
    # Pass 1: singletons
    counts = defaultdict(int)
    for b in baskets:
        for item in b:
            counts[frozenset([item])] += 1
    L = {fs: c for fs, c in counts.items() if c >= min_support}
    freq.update(L)

    k = 2
    while L:
        candidates = set()
        prev = list(L.keys())
        for i in range(len(prev)):
            for j in range(i+1, len(prev)):
                u = prev[i] | prev[j]
                if len(u) == k:
                    if all(frozenset(s) in L for s in combinations(u, k-1)):
                        candidates.add(u)
        if not candidates:
            break
        counts = defaultdict(int)
        for b in baskets:
            for c in candidates:
                if c <= b:
                    counts[c] += 1
        L = {fs: cnt for fs, cnt in counts.items() if cnt >= min_support}
        freq.update(L)
        k += 1

    return freq


def son(baskets, global_support, n_partitions=None):
    """
    SON (Savasere-Omiecinski-Navathe) algorithm.

    Inputs
    ------
    baskets          : list of sets
    global_support   : int s — global frequency threshold
    n_partitions     : int p — number of equal-sized chunks (default sqrt(n))

    Output
    ------
    frequent_global : dict {frozenset: int} — globally frequent itemsets
    """
    import math
    n = len(baskets)
    p = n_partitions or max(2, int(math.sqrt(n)))
    chunk_size = max(1, n // p)
    local_support = max(1, global_support // p)

    # ── Pass 1: find candidates in each partition ─────────────────────────────
    candidates = set()
    for start in range(0, n, chunk_size):
        chunk = baskets[start:start + chunk_size]
        local_freq = apriori_one_chunk(chunk, min_support=local_support)
        candidates |= set(local_freq.keys())

    # ── Pass 2: count candidates globally ────────────────────────────────────
    global_counts = defaultdict(int)
    for basket in baskets:
        for cand in candidates:
            if cand <= basket:
                global_counts[cand] += 1

    return {fs: cnt for fs, cnt in global_counts.items() if cnt >= global_support}


# ── Demo ──────────────────────────────────────────────────────────────────────
import random; random.seed(0)
items = list("ABCDEFGH")
baskets = [set(random.sample(items, random.randint(2, 5))) for _ in range(200)]

frequent = son(baskets, global_support=40, n_partitions=4)
print(f"Globally frequent itemsets (s=40) found by SON:")
for fs, cnt in sorted(frequent.items(), key=lambda x: (len(x[0]), -x[1])):
    print(f"  {set(fs):<30}  support={cnt}")
